In [1]:
from nuscenes.nuscenes import NuScenes
from nuscenes.utils.data_classes import Box
from nuscenes.utils.geometry_utils import view_points
from pyquaternion import Quaternion
import matplotlib.pyplot as plt
import os
from PIL import Image
import numpy as np

In [2]:
project_folder = os.path.abspath(os.curdir)
print(project_folder)
dataroot = os.path.join(project_folder, "v1.0-trainval08_blobs")
version = "v1.0-trainval"
nusc = NuScenes(version=version, dataroot=dataroot, verbose=True)

c:\Users\Dell\Desktop\Studia\Glider\FirstProject
Loading NuScenes tables for version v1.0-trainval...
23 category,
8 attribute,
4 visibility,
64386 instance,
12 sensor,
10200 calibrated_sensor,
2631083 ego_pose,
68 log,
850 scene,
34149 sample,
2631083 sample_data,
1166187 sample_annotation,
4 map,
Done loading in 98.571 seconds.
Reverse indexing ...
Done reverse indexing in 25.7 seconds.


In [3]:
N_SAMPLES = 1000
selected_samples = nusc.sample[:N_SAMPLES]
selected_sample_tokens = set(s['token'] for s in selected_samples)

In [ ]:
import os
import json
import shutil

# ---- CONFIG ----
output_dir = "./selected_scenes"
TARGET_COUNT = 1000

os.makedirs(output_dir, exist_ok=True)

# ---- OUTPUT CONTAINERS ----
valid_samples = []
valid_sample_data = []
valid_annotations = []
valid_instances = []

seen_instances = set()
seen_sample_data = set()
seen_annotations = set()

# ---- HELPERS ----
def file_exists(path):
    return os.path.exists(path)

def get_sample_valid(sample):
    """Check if all camera images exist for this sample."""
    for sd_token in sample['data'].values():
        sd = nusc.get("sample_data", sd_token)

        calib = nusc.get("calibrated_sensor", sd['calibrated_sensor_token'])
        sensor = nusc.get("sensor", calib['sensor_token'])

        if sensor['modality'] == 'camera':
            path = os.path.join(nusc.dataroot, sd['filename'])
            if not file_exists(path):
                return False

    return True

def simplify_sample(sample):
    return {
        "token": sample["token"],
        "timestamp": sample["timestamp"],
        "prev": sample["prev"],
        "next": sample["next"],
        "scene_token": sample["scene_token"],
    }

def save_json(name, data):
    path = os.path.join(output_dir, name)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)


# ---- MAIN LOOP ----
for sample in nusc.sample:  # IMPORTANT: iterate over dict entries
    if len(valid_samples) >= TARGET_COUNT:
        break

    sample_token = sample["token"]

    if not get_sample_valid(sample):
        continue

    # ---- SAMPLE ----
    valid_samples.append(simplify_sample(sample))

    # ---- SAMPLE DATA ----
    for sd_token in sample['data'].values():
        if sd_token in seen_sample_data:
            continue

        sd = nusc.get("sample_data", sd_token)
        valid_sample_data.append(sd)
        seen_sample_data.add(sd_token)

        # ---- COPY CAMERA FILES ----
        calib = nusc.get("calibrated_sensor", sd['calibrated_sensor_token'])
        sensor = nusc.get("sensor", calib['sensor_token'])

        if sensor['modality'] == 'camera':
            src = os.path.join(nusc.dataroot, sd['filename'])
            dst = os.path.join(output_dir, sd['filename'])

            os.makedirs(os.path.dirname(dst), exist_ok=True)

            if os.path.exists(src):
                shutil.copy(src, dst)
            else:
                print("Missing file (unexpected):", src)

    # ---- ANNOTATIONS ----
    for ann_token in sample['anns']:
        if ann_token in seen_annotations:
            continue

        ann = nusc.get("sample_annotation", ann_token)
        valid_annotations.append(ann)
        seen_annotations.add(ann_token)

        # ---- INSTANCE ----
        inst_token = ann['instance_token']
        if inst_token not in seen_instances:
            inst = nusc.get("instance", inst_token)
            valid_instances.append(inst)
            seen_instances.add(inst_token)


# ---- SAVE JSON FILES ----
save_json("sample.json", valid_samples)
save_json("sample_data.json", valid_sample_data)
save_json("sample_annotation.json", valid_annotations)
save_json("instance.json", valid_instances)

print("Done. Samples collected:", len(valid_samples))

Done. Samples collected: 1000
